# Task 20: YOLO-OBB 4-Channel Training Orchestrator
This notebook trains the CYCLO-NEXUS Phase 2 Vision Model.

In [ ]:
!pip install ultralytics torch torchvision numpy

In [ ]:
from google.colab import drive
import sys

# Mount Google Drive to access the CYCLO-NEXUS codebase and datasets
drive.mount('/content/drive')

# Append project root to sys.path so we can import our custom modules
sys.path.append('/content/drive/MyDrive/cyclonegaurd')

In [ ]:
import torch
from ultralytics import YOLO
from vision.models.yolo_4ch_adapter import YOLO4ChannelSurgery

# 1. Load a pre-trained YOLO OBB model (e.g., YOLOv8n-obb)
# We start with 3-channel ImageNet weights to leverage learned edge detectors
model = YOLO('yolov8n-obb.pt')

# 2. Perform Weight Surgery for 4-Channel Satellite Data
surgeon = YOLO4ChannelSurgery()

# Locate the first convolutional layer in the Ultralytics backbone
first_conv_layer = model.model.model[0].conv

# Replace it with our surgically adapted 4-channel layer
model.model.model[0].conv = surgeon.adapt_first_conv_layer(first_conv_layer)

print("Weight surgery complete: Model architecture updated for 4-channel multi-spectral inputs.")

In [ ]:
from vision.preprocessing.physics_aware_augment import PhysicsAwareAugmenter

# 3. Configure the Physics-Aware Augmentation Pipeline
# We inject our custom augmenter into the training loop to enforce Coriolis constraints (no flips).
augmenter = PhysicsAwareAugmenter()

print("Physics-aware augmenter initialized. Ready to intercept dataloader transforms.")

# 4. Execute the Training Loop
# Training hyperparameters optimized for T4/A100 Colab GPUs
# Note: In a production run, we would dynamically override the Ultralytics dataset transforms here.
results = model.train(
    data='/content/drive/MyDrive/cyclonegaurd/datasets/nio_cyclones.yaml', # Path to dataset config
    epochs=100,
    imgsz=1024,
    batch=16,          # 16 for A100, reduce to 4 for T4 due to 4-channel 1024x1024 footprint
    device=0,          # Train on GPU
    optimizer='AdamW', # AdamW handles regularization better for spatial tasks
    lr0=1e-4,          # Lower initial LR due to surgery on the first layer
    name='cyclo_nexus_obb'
)

In [ ]:
import shutil
from pathlib import Path

# 5. Export trained weights to persistent Drive storage
weights_path = Path('runs/obb/cyclo_nexus_obb/weights/best.pt')
target_dir = Path('/content/drive/MyDrive/cyclonegaurd/weights/')
target_dir.mkdir(parents=True, exist_ok=True)

if weights_path.exists():
    shutil.copy(weights_path, target_dir / 'yolo_4ch_best.pt')
    print("Weights successfully exported to Google Drive.")
else:
    print("Training failed or weights not found.")